# Notebook 01: Data Preparation 
**Goal:** Load the raw Golub dataset and prepare data through feature selection.

In [11]:
import numpy as np
import pandas as pd
import seaborn as sns

# Load (Adjust paths based on your folder structure)
train = pd.read_csv('data_set_ALL_AML_train.csv')
labels_df = pd.read_csv('actual.csv')

gene_names = train['Gene Accession Number']
# Drop non-gene columns
X_train = train.drop(columns=['Gene Accession Number', 'Gene Description'])
# Dropping call column
X_train = X_train.loc[:, ~X_train.columns.str.contains('call')].T
X_train.columns = gene_names.values

# Target variable
y_train = labels_df[labels_df['patient'] <= 38]['cancer'].map({'ALL': 0, 'AML': 1}).values

# print(f"missing values: {X_train.isnull().sum().sum()}") = 0. No missing values.


**Experiment:** Feature Selection

**Method:** SNR (Signal to Noise Ratio)

**Objective:** Top 50 signal Genes for ALL vs AML to be obtained through SNR

In [12]:
def calculate_snr(X,y):
    # Calculating mean (signal) and std (noise) for each class
    group_0 = X[y == 0] # ALL
    group_1 = X[y ==1] # AML
    
    # Calculating mean and std for each class
    mu0, mu1 = group_0.mean(), group_1.mean()
    std0, std1 = group_0.std(), group_1.std()
    
    # Formula: SNR = mean / standard deviation 
    return (mu1 - mu0) / (std0 + std1)

snr_values = calculate_snr(X_train, y_train)
top_50_genes = snr_values.abs().sort_values(ascending=False).head(50).index
# Adding expression for top 100 for ablation testing
top_100_genes = snr_values.abs().sort_values(ascending=False).head(100).index
# Adding expression for top 5 genes
top_5_genes = snr_values.abs().sort_values(ascending=False).head(5).index

print (f"Top 50 genes based on SNR: {top_50_genes.tolist()}")
print (f"Top 5 genes: {top_50_genes[:5].tolist()}")

Top 50 genes based on SNR: ['M55150_at', 'U50136_rna1_at', 'X95735_at', 'U22376_cds2_s_at', 'M16038_at', 'M23197_at', 'M84526_at', 'Y12670_at', 'U82759_at', 'D49950_at', 'X59417_at', 'M27891_at', 'X17042_at', 'U05259_rna1_at', 'U12471_cds1_at', 'U46751_at', 'M92287_at', 'Y00787_s_at', 'L08246_at', 'X74262_at', 'M80254_at', 'L13278_at', 'M62762_at', 'M31211_s_at', 'U09087_s_at', 'M81933_at', 'M96326_rna1_at', 'M28130_rna1_s_at', 'D26156_s_at', 'M63138_at', 'M11147_at', 'M57710_at', 'M31523_at', 'U32944_at', 'M81695_s_at', 'L47738_at', 'M28170_at', 'X85116_rna1_s_at', 'Z15115_at', 'M19045_f_at', 'X52142_at', 'X15949_at', 'X04085_rna1_at', 'AF009426_at', 'M21551_rna1_at', 'M69043_at', 'S50223_at', 'X14008_rna1_f_at', 'M91432_at', 'M83652_s_at']
Top 5 genes: ['M55150_at', 'U50136_rna1_at', 'X95735_at', 'U22376_cds2_s_at', 'M16038_at']


**Observation:**

 SNR successfully identified top 50 and top 5 gene biomarkers which includes 'X95735_at' (Zyxin) and is in line with the original 1999 Golub Leukemia classification study. This confirms that the feature selection logic is sound.

In [13]:
# Saving cleaned features and variables
X_train.to_csv('X_train_cleaned.csv', index=False)
np.save('y_train.npy', y_train) 

# Saving top 50 genes list for later use
import json
with open('top_50_genes.json', 'w') as f:
    json.dump(top_50_genes.tolist(), f)
with open('top_100_genes.json', 'w') as f1:
    json.dump(top_100_genes.tolist(), f1)
with open('top_5_genes.json', 'w') as f2:
    json.dump(top_5_genes.tolist(), f2)

Preparing notebook_1 items (training data, target variable, and top 50 genes) for further use in experiments.

In [16]:
from datetime import datetime

new_entry = {
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M"),
    'experiment_id': 'SNR_Feature_Selection',
    'data_version': 'top_50_snr',
    'model_type': 'N/A (EDA)',
    'accuracy': 'N/A',
    'f1_score': 'N/A', 
    'notes': f"Successfully identified top markers. Top 5: {list(top_50_genes[:5])}. Signal matches original paper."
}

log_df = pd.read_csv('experiment_log.csv')
log_df = pd.concat([log_df, pd.DataFrame([new_entry])], ignore_index=True)
log_df.to_csv('experiment_log.csv', index=False)